[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/06_linear_regression/07_regularization_ridge_lasso.ipynb)

# Bài 07 — Regularization: Ridge, Lasso, Elastic Net

## 1. Vấn đề: Overfitting và Underfitting

- **Overfitting** (high variance): model quá khớp tập train, dự đoán kém trên dữ liệu mới — loss train giảm nhưng loss test tăng.
- **Underfitting** (high bias): model quá đơn giản, sai nhiều trên cả train lẫn test.
- **Good fit**: khái quát được cả train lẫn test.

<img src="images/Bias-Variance.png" style="width:40%;height:40%;">

Ví dụ dự đoán giá nhà theo diện tích:

<img src="images/overfitting_1.png" style="width:60%;height:60%;">

Một số kỹ thuật khắc phục overfitting: **cross-validation** để chọn model (chương 15), **early stopping**, và **regularization** — chủ đề của bài này (chi tiết hơn ở chương 05, bài 04).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/06_linear_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 2. Regularization

Ý tưởng: thêm vào hàm mất mát một số hạng **phạt độ phức tạp** của model → **regularized loss function**.

### 2.1 Trực giác

Giả sử $h_\theta(x) = \theta_0 + \theta_1 x + \theta_2 x^2 + \theta_3 x^3 + \theta_4 x^4$ bị overfitting (đường xanh dương) và ta muốn "loại bỏ" ảnh hưởng của $\theta_3 x^3 + \theta_4 x^4$ (đường hồng). Ta có thể sửa cost function:

$$\frac{1}{2m}\sum_{i=1}^{m}\left(h_\theta(x^{(i)}) - y^{(i)}\right)^2 + 1000\,\theta_3^2 + 1000\,\theta_4^2$$

Khi tối ưu, $\theta_3, \theta_4$ bị ép về gần 0 → model gần như bậc 2.

<img src="images/reg_1.png" style="width:35%;height:35%;">

> 📝 *Đính chính:* bản gốc viết $h_\theta(x^{(i)} - y^{(i)})$ — dấu ngoặc đúng là $(h_\theta(x^{(i)}) - y^{(i)})$.

### 2.2 L2 Regularization (Ridge)

Thay vì chọn tay từng tham số, phạt **mọi** trọng số bằng tổng bình phương (chuẩn L2):

$$\lVert\theta\rVert_2^2 = \theta_1^2 + \theta_2^2 + \dots + \theta_n^2$$

$$J(\theta) = \frac{1}{2m}\left[\sum_{i=1}^{m}\left(h_\theta(x^{(i)}) - y^{(i)}\right)^2 + \lambda\sum_{j=1}^{n}\theta_j^2\right]$$

- $\lambda$ là **hệ số regularization** — hyperparameter điều khiển độ mạnh của việc phạt. Thử trực quan tại [desmos](https://www.desmos.com/calculator/1hexc8ntqp).
- **Không phạt $\theta_0$** (bias): bias chỉ dịch chuyển dự đoán lên/xuống, không làm model phức tạp hơn.

### 2.3 Gradient descent cho Regularized Linear Regression

$$\theta_0 := \theta_0 - \alpha\frac{1}{m}\sum_{i=1}^m\left(h_\theta(x^{(i)}) - y^{(i)}\right)$$

$$\theta_j := \theta_j - \alpha\frac{1}{m}\left[\sum_{i=1}^m\left(h_\theta(x^{(i)}) - y^{(i)}\right)x_j^{(i)} + \lambda\theta_j\right], \quad j = 1, \dots, n$$

Viết lại số hạng của $\theta_j$: $\theta_j := \theta_j\left(1 - \alpha\frac{\lambda}{m}\right) - \alpha\frac1m\sum_i(\dots)x_j^{(i)}$ — mỗi bước **co** trọng số lại một chút → tên gọi **weight decay** trong deep learning.

**Vector hoá:** $\theta := \theta - \frac{\alpha}{m}\left[X^\top(X\theta - \vec y) + \lambda\theta'\right]$, với $\theta'$ là $\theta$ nhưng phần tử đầu ($\theta_0$) bằng 0.

**Nghiệm đóng** *(bổ sung)*: $\theta = (X^\top X + \lambda I')^{-1}X^\top y$ ($I'$ là ma trận đơn vị với phần tử đầu bằng 0) — luôn khả nghịch khi $\lambda > 0$, khắc phục đa cộng tuyến (bài 03).

In [ ]:
def ridge_gd(X, y, lam, alpha=0.1, iters=3000):
    """X đã có cột bias ở đầu. Không phạt theta_0."""
    m = len(y)
    theta = np.zeros(X.shape[1])
    for _ in range(iters):
        reg = lam * theta
        reg[0] = 0.0
        theta -= alpha / m * (X.T @ (X @ theta - y) + reg)
    return theta


def ridge_closed(X, y, lam):
    I = np.eye(X.shape[1])
    I[0, 0] = 0.0
    return np.linalg.solve(X.T @ X + lam * I, X.T @ y)


x = np.sort(rng.uniform(-3, 3, 30))
y = np.sin(1.5 * x) + rng.normal(0, 0.3, 30)
Phi = StandardScaler().fit_transform(PolynomialFeatures(10, include_bias=False).fit_transform(x[:, None]))
Xb = np.c_[np.ones(len(x)), Phi]
print("GD vs nghiệm đóng khớp nhau:", np.allclose(ridge_gd(Xb, y, lam=5.0), ridge_closed(Xb, y, 5.0), atol=1e-3))

## 3. Ảnh hưởng của λ

In [ ]:
grid = np.linspace(-3, 3, 300)
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, lam in zip(axes, [0.0, 0.1, 10, 1000]):
    model = make_pipeline(PolynomialFeatures(10, include_bias=False), StandardScaler(),
                          Ridge(alpha=lam) if lam > 0 else LinearRegression()).fit(x[:, None], y)
    coef = model[-1].coef_
    ax.scatter(x, y, s=15)
    ax.plot(grid, model.predict(grid[:, None]), "r", lw=2)
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f"λ = {lam}: ‖w‖ = {np.linalg.norm(coef):.1f}")
plt.show()

λ = 0 → overfit (trọng số khổng lồ); λ vừa → khớp tốt; λ quá lớn → underfit (trọng số gần 0, gần như đường thẳng nằm ngang).

> ⚠️ Trong scikit-learn, $\lambda$ được gọi là **`alpha`** (`Ridge(alpha=...)`) — đừng nhầm với learning rate $\alpha$ của tài liệu Andrew Ng/AIVN! Và scikit-learn dùng loss **tổng** (không chia $m$) cho Ridge: $\lVert y - Xw\rVert^2 + \alpha\lVert w\rVert^2$.

## 4. L1 Regularization (Lasso) và Elastic Net *(bổ sung)*

**Lasso** phạt tổng **trị tuyệt đối**:

$$J(\mathbf w) = \frac{1}{2n}\lVert y - X\mathbf w\rVert^2 + \lambda\lVert\mathbf w\rVert_1$$

Tính chất nổi bật: Lasso đẩy nhiều hệ số về **đúng bằng 0** → tự động **chọn feature** (sparse model). Lasso không có nghiệm đóng; giải bằng **coordinate descent** với phép soft-thresholding (xem `LassoScratch` trong `src/ml_course/algorithms/linear_models.py`).

**Elastic Net** kết hợp cả hai: $\lambda\left[\rho\lVert\mathbf w\rVert_1 + \frac{1-\rho}{2}\lVert\mathbf w\rVert_2^2\right]$ — giữ khả năng chọn feature của Lasso nhưng ổn định hơn khi các feature tương quan với nhau.

### Vì sao L1 cho nghiệm thưa? — góc nhìn hình học

Bài toán có regularization tương đương: cực tiểu MSE **với ràng buộc** $\lVert\mathbf w\rVert \le t$. Đường đồng mức của MSE (elip) chạm vào vùng ràng buộc:
- L2: vùng ràng buộc là **hình tròn** → điểm chạm thường không nằm trên trục.
- L1: vùng ràng buộc là **hình thoi** có **góc nhọn trên các trục** → điểm chạm rất hay rơi vào **góc** → một hệ số bằng 0.

In [ ]:
w1, w2 = np.meshgrid(np.linspace(-1.5, 3.5, 300), np.linspace(-1.5, 3.5, 300))
loss = (w1 - 2.2) ** 2 + 3 * (w2 - 1.2) ** 2 + 1.6 * (w1 - 2.2) * (w2 - 1.2)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, (name, region, opt) in zip(axes, [("L2 (Ridge)", w1**2 + w2**2 <= 1, (0.83, 0.56)),
                                          ("L1 (Lasso)", np.abs(w1) + np.abs(w2) <= 1, (1.0, 0.0))]):
    ax.contour(w1, w2, loss, levels=15, cmap="viridis")
    ax.contourf(w1, w2, region, levels=[0.5, 1], colors=["#fecaca"], alpha=0.8)
    ax.plot(2.2, 1.2, "k*", ms=14, label="nghiệm không phạt (OLS)")
    ax.plot(*opt, "ro", ms=9, label="nghiệm có ràng buộc (minh hoạ)")
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)
    ax.set_aspect("equal")
    ax.set_title(name)
    ax.legend(loc="lower left", fontsize=8)
plt.show()

### Coefficient path: hệ số thay đổi theo λ

Dữ liệu có 10 feature, chỉ **3 feature thật sự có ích**:

In [ ]:
n, d = 200, 10
Xr = rng.normal(size=(n, d))
true_w = np.array([3.0, -2.0, 1.5] + [0.0] * 7)
yr = Xr @ true_w + rng.normal(0, 1.0, n)
lams = np.logspace(-3, 1.5, 60)

ridge_path = np.array([Ridge(alpha=l * n).fit(Xr, yr).coef_ for l in lams])     # nhân n để cùng thang với Lasso của sklearn
lasso_path = np.array([Lasso(alpha=l).fit(Xr, yr).coef_ for l in lams])
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (name, path) in zip(axes, [("Ridge", ridge_path), ("Lasso", lasso_path)]):
    for j in range(d):
        ax.plot(lams, path[:, j], lw=2 if true_w[j] != 0 else 1, color="#2563eb" if true_w[j] != 0 else "gray")
    ax.set_xscale("log")
    ax.set_xlabel("λ")
    ax.set_title(f"{name}: xanh = feature có ích, xám = nhiễu")
    ax.axhline(0, color="k", lw=0.5)
plt.show()

- **Ridge** co **mọi** hệ số về gần 0 một cách mượt mà, nhưng hiếm khi bằng đúng 0.
- **Lasso** đưa hệ số của các feature nhiễu về **đúng 0** rất sớm; với λ phù hợp chỉ còn lại đúng 3 feature có ích → model gọn, dễ diễn giải.

In [ ]:
from ml_course.algorithms import LassoScratch

lasso_sk = Lasso(alpha=0.1).fit(Xr, yr)
lasso_ours = LassoScratch(alpha=0.1).fit(Xr, yr)
print("sklearn Lasso :", lasso_sk.coef_.round(3))
print("LassoScratch  :", lasso_ours.coef_.round(3))
print("Số hệ số khác 0:", np.sum(lasso_sk.coef_ != 0), "/", d)

## 5. Chọn λ bằng cross-validation và quy tắc thực hành

In [ ]:
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV

ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 50)).fit(Xr, yr)
lasso_cv = LassoCV(cv=5, random_state=0).fit(Xr, yr)
enet_cv = ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8, 1.0], cv=5, random_state=0).fit(Xr, yr)
print(f"RidgeCV      : alpha = {ridge_cv.alpha_:.3f}")
print(f"LassoCV      : alpha = {lasso_cv.alpha_:.4f}, số feature giữ lại = {np.sum(lasso_cv.coef_ != 0)}")
print(f"ElasticNetCV : alpha = {enet_cv.alpha_:.4f}, l1_ratio = {enet_cv.l1_ratio_}")

| Tình huống | Gợi ý |
|---|---|
| Nhiều feature đều có chút ích, tương quan với nhau | **Ridge** |
| Tin rằng chỉ ít feature quan trọng, muốn chọn feature | **Lasso** |
| Nhiều feature tương quan + muốn chọn feature | **Elastic Net** (Lasso chọn ngẫu nhiên một trong nhóm feature tương quan) |
| Nhiều feature hơn số mẫu ($d > n$) | Ridge / Elastic Net (Lasso chọn tối đa $n$ feature) |

**Bắt buộc:** chuẩn hoá feature trước khi regularization — nếu không, feature có thang đo nhỏ (cần hệ số lớn) bị phạt nặng một cách bất công. Đặt scaler trong `Pipeline`.

## 6. Regularized Logistic Regression

Cùng ý tưởng áp dụng cho Logistic Regression (chương 07): thêm $\frac{\lambda}{2m}\sum_{j=1}^n\theta_j^2$ vào cross-entropy; cập nhật gradient có cùng dạng $+\frac{\lambda}{m}\theta_j$.

## 🧠 Kiểm tra nhanh

1. Vì sao không phạt $\theta_0$?
2. Tăng λ thì bias và variance thay đổi thế nào?
3. Vì sao Lasso cho nhiều hệ số bằng đúng 0 còn Ridge thì không?
4. `Ridge(alpha=10)` trong sklearn — `alpha` ở đây là learning rate hay hệ số regularization?

## Tài liệu tham khảo

[1] [Underfitting and Overfitting in Machine Learning](https://www.geeksforgeeks.org/underfitting-and-overfitting-in-machine-learning/) · [2] [Overfitting — Machine Learning Cơ Bản](https://machinelearningcoban.com/2017/03/04/overfitting/) · [3] Tibshirani (1996) — *Regression Shrinkage and Selection via the Lasso* · [4] ISLR, chương 6.2

---
<sub>Bài gốc: *Regularization* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 4 (MIT License) — phần Linear Regression. Đã sửa lỗi dấu ngoặc trong công thức; bổ sung: cài đặt GD & nghiệm đóng, ảnh hưởng của λ, Lasso, Elastic Net, hình học L1/L2, coefficient path, chọn λ bằng CV. Phần Regularized Logistic Regression học kỹ ở chương 07.</sub>